# Bài 32: Perceptron nhiều lớp

Nơ-ron đơn lẻ ở Bài 31, dù được huấn luyện cẩn thận đến đâu, cũng không vượt qua được mức đoán ngẫu nhiên trên tập dữ liệu vành-khuyên-bao-quanh-hình-đĩa &mdash; một phép chiếu tuyến tính đơn lẻ đơn giản là không đủ khả năng biểu diễn. Bài này thêm một lớp nữa: thay vì *một* phép chiếu theo sau là một ngưỡng, dùng *hai* phép chiếu với một hàm phi tuyến xen giữa. Chỉ vậy thôi. Đó là toàn bộ ý tưởng đằng sau mọi mạng sâu trong khóa học này &mdash; xếp chồng thêm nhiều lớp như thế.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

## Lại là bài toán không giải được

In [ ]:
rng = np.random.default_rng(0)
theta_in = rng.uniform(0, 2 * np.pi, 60)
inner = np.stack([0.5 * np.cos(theta_in), 0.5 * np.sin(theta_in)], axis=1) + rng.normal(0, 0.1, (60, 2))
theta_out = rng.uniform(0, 2 * np.pi, 60)
outer = np.stack([2.0 * np.cos(theta_out), 2.0 * np.sin(theta_out)], axis=1) + rng.normal(0, 0.15, (60, 2))
X = np.vstack([inner, outer])
y = np.concatenate([np.zeros(60), np.ones(60)])

plt.scatter(*inner.T, s=15, label='inner (y=0)')
plt.scatter(*outer.T, s=15, label='outer (y=1)')
plt.legend(fontsize=8)
plt.gca().set_aspect('equal')
plt.title('No single linear projection separates these (Lessons 30-31)')
plt.show()

## Hai phép chiếu, một hàm phi tuyến

Một **perceptron nhiều lớp (multi-layer perceptron — MLP)** nối tiếp một phép chiếu *ẩn* vào một không gian mới, một **hàm kích hoạt (activation)** phi tuyến áp dụng theo từng phần tử, rồi một phép chiếu tuyến tính cuối cùng (chính là nơ-ron ở Bài 31) bên trên các tọa độ *đã biến đổi*:

$$h = \text{ReLU}(W_1 x + b_1), \qquad z = w_2^\top h + b_2, \qquad p = \sigma(z)$$

$\text{ReLU}(u) = \max(0, u)$ là hàm kích hoạt phổ biến đơn giản nhất: nó tuyến tính ở mọi nơi trừ một điểm gãy duy nhất tại 0. Một điểm gãy đó, áp dụng độc lập lên mọi đơn vị của $h$, là đủ &mdash; lớp ẩn có thể uốn, gập và kéo giãn không gian đầu vào sao cho một ranh giới *tuyến tính* trong không gian đã biến đổi tương ứng với một ranh giới phi tuyến cao khi quay lại tọa độ ban đầu.

ReLU và sigmoid là hai hàm kích hoạt khác nhau làm các việc khác nhau ở đây: ReLU là hàm phi tuyến của *lớp ẩn*, định hình lại không gian đầu vào để các điểm dễ tách hơn, trong khi sigmoid (giữ nguyên từ Bài 31) nén điểm số cuối cùng $z$ thành một xác suất ở cuối. Hình dạng của chúng trông khá giống nhau (đồ thị bên dưới), nhưng mỗi hàm phù hợp với vai trò riêng: đầu ra không bị chặn và đạo hàm đơn giản của ReLU hoạt động tốt khi nằm sâu bên trong mạng, trong khi đầu ra bị giới hạn trong (0, 1) của sigmoid khiến nó đọc được như một xác suất ở đầu ra.

In [ ]:
zz = np.linspace(-5, 5, 200)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(zz, np.maximum(0, zz))
axes[0].set_title('ReLU (hidden-layer activation)')
axes[0].set_xlabel('u')
axes[0].set_ylabel('ReLU(u)')
axes[1].plot(zz, 1 / (1 + np.exp(-zz)))
axes[1].set_title('sigmoid (output activation)')
axes[1].set_xlabel('z')
axes[1].set_ylabel('sigma(z)')
plt.tight_layout()
plt.show()

### Tại sao hàm phi tuyến là thiết yếu

Không có ReLU &mdash; nếu lớp ẩn được để nguyên là $h = W_1 x + b_1$ &mdash; hai lớp sẽ sụp đổ thành một phép chiếu tuyến tính duy nhất:

$$z = w_2^\top(W_1 x + b_1) + b_2 = \underbrace{(w_2^\top W_1)}_{w'^\top} x + \underbrace{(w_2^\top b_1 + b_2)}_{b'}$$

Đó chính xác lại là nơ-ron đơn lẻ ở Bài 31, chỉ là với trọng số của hai lớp được nhân với nhau thành một $w'$ và $b'$. Đây là mặt trái của nhận xét ở Bài 30 rằng phép nhân ma trận là phép chiếu xếp chồng: ghép hai phép nhân ma trận *vẫn* chỉ là một phép nhân ma trận, nên xếp chồng các lớp thuần tuyến tính chẳng được gì, bất kể bạn nối bao nhiêu lớp. Điểm gãy của ReLU là thứ phá vỡ sự sụp đổ đó: vì nó không phải hàm tuyến tính, $w_2^\top \text{ReLU}(W_1x+b_1)+b_2$ không thể viết lại thành $w'^\top x + b'$.

In [ ]:
demo_rng = np.random.default_rng(3)
W1_demo = demo_rng.normal(size=(2, 4))
b1_demo = demo_rng.normal(size=4)
W2_demo = demo_rng.normal(size=(4, 1))
b2_demo = demo_rng.normal(size=1)

z_two_linear_layers = (X @ W1_demo + b1_demo) @ W2_demo + b2_demo   # no nonlinearity in between

W_combined = W1_demo @ W2_demo                # a single (2, 1) matrix
b_combined = b1_demo @ W2_demo + b2_demo      # a single offset
z_one_combined_layer = X @ W_combined + b_combined

print(f'max diff, two LINEAR layers vs. one combined layer: '
      f'{np.abs(z_two_linear_layers - z_one_combined_layer).max():.2e}')

## Lan truyền ngược qua hai lớp

Quy tắc chuỗi mở rộng một cách gọn gàng: lan truyền tín hiệu sai số $\partial L/\partial z$ từ Bài 31 ngược qua phép chiếu đầu ra để có $\partial L/\partial h$, rồi ngược qua ReLU và phép chiếu ẩn để có $\partial L/\partial W_1, \partial L/\partial b_1$. Mỗi bước hoặc là "nhân với chuyển vị của một ma trận trọng số" hoặc là "nhân từng phần tử với đạo hàm của một hàm kích hoạt" &mdash; xếp chồng thêm lớp chỉ là lặp lại hai phép toán này xa hơn về phía sau qua chuỗi.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(np.float64)

def forward(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    a1 = relu(z1)
    z2 = (a1 @ W2 + b2).ravel()
    p = sigmoid(z2)
    return p, (z1, a1, z2)

def backward(X, y, p, cache, W2):
    z1, a1, z2 = cache
    n = len(y)
    grad_z2 = ((p - y) / n).reshape(-1, 1)
    grad_W2 = a1.T @ grad_z2
    grad_b2 = grad_z2.sum(axis=0)
    grad_a1 = grad_z2 @ W2.T
    grad_z1 = grad_a1 * relu_deriv(z1)
    grad_W1 = X.T @ grad_z1
    grad_b1 = grad_z1.sum(axis=0)
    return grad_W1, grad_b1, grad_W2, grad_b2

### Đối chiếu với autograd của PyTorch

In [ ]:
H = 4
init_rng = np.random.default_rng(8)
W1 = init_rng.normal(size=(2, H)) * 0.7
b1 = np.zeros(H)
W2 = init_rng.normal(size=(H, 1)) * 0.7
b2 = np.zeros(1)

p, cache = forward(X, W1, b1, W2, b2)
grad_W1, grad_b1, grad_W2, grad_b2 = backward(X, y, p, cache, W2)

X_t = torch.tensor(X)
y_t = torch.tensor(y)
W1_t = torch.tensor(W1, requires_grad=True)
b1_t = torch.tensor(b1, requires_grad=True)
W2_t = torch.tensor(W2, requires_grad=True)
b2_t = torch.tensor(b2, requires_grad=True)

z1_t = X_t @ W1_t + b1_t
a1_t = torch.relu(z1_t)
z2_t = (a1_t @ W2_t + b2_t).squeeze(-1)
loss_t = torch.nn.functional.binary_cross_entropy_with_logits(z2_t, y_t)
loss_t.backward()

print(f'W1 max diff: {np.abs(grad_W1 - W1_t.grad.numpy()).max():.2e}')
print(f'W2 max diff: {np.abs(grad_W2 - W2_t.grad.numpy()).max():.2e}')
print(f'b1 max diff: {np.abs(grad_b1 - b1_t.grad.numpy()).max():.2e}')
print(f'b2 max diff: {np.abs(grad_b2 - b2_t.grad.numpy()).max():.2e}')

## Huấn luyện

Cùng công thức như Bài 31 -- lượt xuôi, hàm mất mát, lượt ngược, bước ngược chiều gradient -- chỉ là với bốn tensor tham số ($W_1, b_1, W_2, b_2$) được cập nhật mỗi epoch thay vì hai. Hàm mất mát tiếp tục giảm về phía (nhưng không bao giờ chạm đúng) 0 ngay cả sau khi độ chính xác đạt 100%, vì các dự đoán tự-tin-và-đúng luôn có thể tự tin thêm một chút nữa.

### Quan sát vòng lặp chạy, từng giai đoạn một

Ảnh động bên dưới chạy qua cùng công thức 4 giai đoạn &mdash; lượt xuôi, hàm mất mát, lượt ngược, cập nhật &mdash; giờ qua hai đơn vị ẩn thay vì một, trên một ví dụ chọn thủ công (không phải tập dữ liệu 120 điểm thật bên dưới): $x=(2, 1)$, $y=1$, các đơn vị ẩn $w_0=(0.2,-0.4)$, $b_0=0.1$ và $w_1=(-0.3,0.5)$, $b_1=0.3$, lớp đầu ra $w_2=(0.4,0.5)$, $b_2=-0.6$, $\eta=0.5$. Giống phiên bản ở Bài 31, các điểm kiểm tra là các bước liên tiếp (không ngắt quãng), nên mọi con số đều có thể tái tạo bằng tay.

<img src="../animations/mlp_training_loop_demo.gif" alt="Ảnh động vòng lặp huấn luyện 2 đơn vị ẩn: một đồ thị tính toán với hai khối tuyến tính+ReLU song song dẫn vào một khối tuyến tính đầu ra chung, sigmoid và khối hàm mất mát, luân phiên qua các giai đoạn lượt xuôi, hàm mất mát, lượt ngược và cập nhật kèm thanh tiến trình, trên một ví dụ tính tay" style="display: block; margin: 0 auto; max-width: 900px;">

**Tự tính tay bước đầu tiên:**

$$z_0 = w_0\cdot x+b_0 = (0.2)(2)+(-0.4)(1)+0.1=0.1, \qquad h_0=\text{ReLU}(z_0)=0.1$$
$$z_1 = w_1\cdot x+b_1 = (-0.3)(2)+(0.5)(1)+0.3=0.2, \qquad h_1=\text{ReLU}(z_1)=0.2$$
$$z_2 = w_2\cdot h+b_2 = (0.4)(0.1)+(0.5)(0.2)-0.6=-0.46$$
$$p=\sigma(-0.46)\approx0.387, \qquad L=-\log(p)\approx0.949$$

Lượt ngược, tính cho đơn vị ẩn 0 (đơn vị 1 và lớp đầu ra theo đúng cùng mẫu &mdash; để lại như bài tập):

$$\frac{\partial L}{\partial z_2}=p-y\approx-0.613$$
$$\frac{\partial L}{\partial h_0}=\frac{\partial L}{\partial z_2}\cdot w_{2,0}\approx-0.245, \qquad \frac{\partial L}{\partial z_0}=\frac{\partial L}{\partial h_0}\cdot\text{ReLU}'(z_0)\approx-0.245 \quad(\text{since } z_0>0)$$
$$\frac{\partial L}{\partial w_{0,1}}=x_1\cdot\frac{\partial L}{\partial z_0}\approx-0.490, \qquad \frac{\partial L}{\partial w_{0,2}}\approx-0.245, \qquad \frac{\partial L}{\partial b_0}\approx-0.245$$
$$w_{0,1} \leftarrow 0.2-0.5(-0.490)\approx0.445, \qquad w_{0,2} \leftarrow -0.4-0.5(-0.245)\approx-0.277, \qquad b_0 \leftarrow 0.1-0.5(-0.245)\approx0.223$$

Đó chính xác là điều ảnh động cho thấy sau lần cập nhật đầu tiên.

#### Quay lại tập dữ liệu thật

Khi cơ chế hai đơn vị ẩn đã được xác nhận bằng tay, phần còn lại của mục này quay lại mạng thật: $H=4$ đơn vị ẩn, được huấn luyện trên toàn bộ tập dữ liệu vành-khuyên-và-đĩa ở phần trước của bài.

In [ ]:
lr = 0.1
losses = []
for epoch in range(3000):
    p, cache = forward(X, W1, b1, W2, b2)
    eps = 1e-9
    losses.append(-np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps)))
    grad_W1, grad_b1, grad_W2, grad_b2 = backward(X, y, p, cache, W2)
    W1 -= lr * grad_W1; b1 -= lr * grad_b1
    W2 -= lr * grad_W2; b2 -= lr * grad_b2

final_pred = (p > 0.5).astype(float)
print(f'final accuracy: {(final_pred == y).mean():.1%}  (cf. single neuron from Lesson 31 managed 50%)')
print(f'final loss: {losses[-1]:.4f}')

plt.plot(losses)
plt.xlabel('epoch'); plt.ylabel('loss')
plt.title('Training loss (2-layer MLP)')
plt.show()

## Các phép chiếu của lớp ẩn, nhìn thấy được

Mỗi đơn vị ẩn tự nó là một phép chiếu kiểu Bài 30: một hướng $w_i$ (một cột của $W_1$) và một độ lệch $b_i$ (một phần tử của $b_1$), cùng nhau xác định một đường cắt $w_i \cdot x + b_i = 0$ với ReLU đưa mọi thứ ở một phía của nó về 0. Với chỉ 2 chiều đầu vào, cả $H$ đường cắt này có thể được vẽ trực tiếp lên dữ liệu ban đầu.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(*X[y == 0].T, s=15, label='inner')
ax.scatter(*X[y == 1].T, s=15, label='outer')

colors = plt.cm.tab10.colors
for i in range(W1.shape[1]):
    w_i, b_i = W1[:, i], b1[i]
    # the cut {x : w_i.x + b_i = 0}, drawn through its closest point to the origin
    foot = -b_i * w_i / (w_i @ w_i)
    perp = np.array([-w_i[1], w_i[0]])
    p1, p2 = foot + 5 * perp, foot - 5 * perp
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color=colors[i], linewidth=1.5, label=f'hidden unit {i}')
    ax.arrow(*foot, *(w_i / np.linalg.norm(w_i) * 0.5), head_width=0.1, color=colors[i], length_includes_head=True)

ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
ax.set_aspect('equal')
ax.legend(fontsize=7, loc='upper right')
ax.set_title('Each hidden unit is one projection direction $w_i$,\ncutting the space along $w_i \\cdot x + b_i = 0$')
plt.tight_layout()
plt.show()

Tự mỗi đường cắt chỉ là một đường thẳng &mdash; không đường nào trong số đó tách được vành khuyên khỏi hình đĩa. Nhưng lớp đầu ra kết hợp tất cả $H$ kết quả (đã qua ReLU) của chúng lại với nhau, và một nhúm đường cắt thẳng xếp quanh vành khuyên, kết hợp với các trọng số phù hợp, có thể xấp xỉ một đường cong khép kín đủ tốt để bao quanh nó. Bước kết hợp đó chính là thứ mà phần còn lại của mục này trực quan hóa.

Ảnh động này chạy toàn bộ vòng lặp huấn luyện, vẽ lại cả bốn đường cắt của các đơn vị ẩn và vùng quyết định kết hợp (được tô bóng) cùng nhau ở mỗi điểm kiểm tra. Ở epoch 0, bốn đường thẳng và vùng tô bóng về cơ bản là tùy ý; đến cuối, bốn đường cắt đã xoay và dịch chuyển sao cho giao tô bóng của chúng là một đa giác chặt ôm sát gần như chính xác hình đĩa bên trong &mdash; vài đường thẳng, kết hợp qua thêm một lớp tuyến tính, uốn thành một đường cong khép kín.

<img src="../animations/mlp_training_demo.gif" alt="Ảnh động huấn luyện MLP trên tập dữ liệu vành-khuyên-và-đĩa: bốn đường cắt thẳng của các đơn vị ẩn và vùng quyết định tô bóng thu được tiến triển cùng với đường cong hàm mất mát, hội tụ về một đa giác chặt bao quanh hình đĩa bên trong" style="display: block; margin: 0 auto; max-width: 760px;">

Khi huấn luyện đã hội tụ, hãy đóng băng các trọng số và xem xét bốn đơn vị ẩn từng cái một. Khung bên trái cho thấy một đơn vị riêng lẻ: đường cắt của nó và nửa mặt phẳng nơi nó "bật" (đầu ra ReLU > 0). Khung bên phải cho thấy lượt xuôi một phần qua lớp đầu ra (đã đóng băng) (đóng góp của các đơn vị chưa được đưa vào bị đưa về 0 trong tổng), nên đến bước cuối nó rơi đúng vào ranh giới quyết định thực tế của mạng.

<img src="../animations/mlp_hidden_units_demo.gif" alt="Ảnh động so sánh bốn đơn vị ẩn của một MLP đã huấn luyện: khung trái cho thấy đường cắt và nửa mặt phẳng hoạt động của mỗi lần một đơn vị ẩn, khung phải cho thấy vùng quyết định tích lũy từ một lượt xuôi một phần thực sự khi các đơn vị được thêm vào lần lượt, kết thúc tại ranh giới quyết định thực tế của mạng" style="display: block; margin: 0 auto; max-width: 760px;">

## Lớp ẩn thực sự đã làm gì

Lớp đầu ra *chỉ* là một phép chiếu tuyến tính (nơ-ron ở Bài 31) &mdash; nhưng nó tác động lên $h$, đầu ra của lớp ẩn, không phải lên $x$ ban đầu. Nếu lớp ẩn làm tốt việc của mình, các điểm *đã biến đổi* sẽ dễ tách bằng một đường thẳng hơn nhiều. Vì $h$ nằm trong $\mathbb{R}^4$ ở đây, ta dùng PCA (Bài 6) để trực quan hóa nó trong 2D.

In [ ]:
_, (_, hidden, _) = forward(X, W1, b1, W2, b2)

hidden_centered = hidden - hidden.mean(axis=0)
cov = np.cov(hidden_centered.T)
eigvals, eigvecs = np.linalg.eigh(cov)
top2_directions = eigvecs[:, -2:]
hidden_2d = hidden_centered @ top2_directions

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].scatter(*X[y == 0].T, s=15, label='inner')
axes[0].scatter(*X[y == 1].T, s=15, label='outer')
axes[0].set_aspect('equal')
axes[0].set_title('Original input space')
axes[0].legend(fontsize=7)

axes[1].scatter(*hidden_2d[y == 0].T, s=15, label='inner')
axes[1].scatter(*hidden_2d[y == 1].T, s=15, label='outer')
axes[1].set_title('Hidden representation\n(top 2 PCA directions of a 4D space)')
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

Ngay cả hình chụp 2D mất mát thông tin này của toàn bộ không gian ẩn 4D cũng cho thấy hai lớp bị kéo ra xa nhau thành gần như tách được tuyến tính &mdash; một cải thiện lớn so với mức trần 74%, vốn là điều tốt nhất mà *bất kỳ* đường thẳng nào làm được trong không gian ban đầu (Bài 30). Nơ-ron đầu ra thực tế làm việc trong toàn bộ không gian ẩn 4D, nơi nó đạt chính xác độ chính xác 100%. Đây là toàn bộ cơ chế của học sâu thu nhỏ: mỗi lớp *định hình lại không gian* để công việc của lớp tiếp theo dễ dàng hơn, cho đến khi công việc của lớp cuối cùng trở nên tầm thường &mdash; một phép chiếu tuyến tính đơn lẻ.

## Năng lực mô hình: bao nhiêu đơn vị ẩn là đủ?

**Năng lực (capacity)** của một mô hình là mức độ phong phú của họ hàm mà nó có thể biểu diễn. Ở đây năng lực được đặt trực tiếp bởi $H$, độ rộng của lớp ẩn: mỗi đơn vị ẩn đóng góp một đường cắt thẳng (kiểu Bài 30), và lớp đầu ra kết hợp các đơn vị này. Quá ít đơn vị, và mô hình đơn giản là không thể biểu diễn ranh giới mà tập dữ liệu cần — không lượng huấn luyện nào có thể tìm ra một nghiệm không tồn tại. Hãy quét $H$ và huấn luyện lại từ đầu ở mỗi độ rộng, qua vài lần khởi tạo ngẫu nhiên, để thấy trực tiếp mức trần.

In [ ]:
def train_and_eval(H, seed, epochs=3000, lr=0.1):
    init_rng = np.random.default_rng(seed)
    W1_s = init_rng.normal(size=(2, H)) * 0.7
    b1_s = np.zeros(H)
    W2_s = init_rng.normal(size=(H, 1)) * 0.7
    b2_s = np.zeros(1)
    for _ in range(epochs):
        p, cache = forward(X, W1_s, b1_s, W2_s, b2_s)
        gW1, gb1, gW2, gb2 = backward(X, y, p, cache, W2_s)
        W1_s -= lr * gW1; b1_s -= lr * gb1
        W2_s -= lr * gW2; b2_s -= lr * gb2
    p, _ = forward(X, W1_s, b1_s, W2_s, b2_s)
    return ((p > 0.5).astype(float) == y).mean()

widths = [1, 2, 3, 4, 6, 8, 16]
seeds = range(5)
mean_accs, min_accs, max_accs = [], [], []
for H_test in widths:
    accs = [train_and_eval(H_test, seed) for seed in seeds]
    mean_accs.append(np.mean(accs)); min_accs.append(np.min(accs)); max_accs.append(np.max(accs))
    print(f'H={H_test:>2}: mean acc={np.mean(accs):.1%}  (min={np.min(accs):.1%}, max={np.max(accs):.1%}, over {len(seeds)} seeds)')

plt.figure(figsize=(5.5, 3.5))
plt.plot(widths, mean_accs, '-o', label='mean')
plt.fill_between(widths, min_accs, max_accs, alpha=0.2, label='min-max range')
plt.xlabel('hidden units ($H$)'); plt.ylabel('final accuracy')
plt.title('Accuracy vs. capacity, across 5 random seeds each')
plt.legend(fontsize=8)
plt.show()

$H=1$ dừng ở mức thấp hơn nhiều so với 100% bất kể huấn luyện bao lâu hay bắt đầu từ seed nào — đó không phải thất bại tối ưu hóa, mà là một mức trần biểu diễn: một đơn vị ẩn chỉ là một đường cắt đơn lẻ, nên nó thừa hưởng gần như chính xác giới hạn tuyến tính ~74% của Bài 30.

$H=2$ tốt hơn, nhưng vẫn bị giới hạn bởi năng lực — không seed nào trong năm seed ở trên đạt 100%.

$H=3$ mới thực sự là lằn ranh mong manh: năng lực để biểu diễn một nghiệm hoàn hảo đã có (một seed ở trên đạt chính xác 100%), nhưng việc huấn luyện có tìm ra nó hay không phụ thuộc vào việc bắt đầu từ seed *nào* (một lời báo trước về các vấn đề địa hình tối ưu hóa ở Bài 33).

Đến $H=4$, mọi seed đều đạt 100% một cách đáng tin cậy: có đủ năng lực để mạng không còn là nút thắt cổ chai, và huấn luyện tìm ra một nghiệm trong phạm vi năng lực đó một cách đáng tin cậy. Đây là hình dạng tổng quát của một đường cong năng lực — một mức sàn cứng được đặt bởi những gì mô hình *có thể* biểu diễn, với độ khó tối ưu hóa chồng lên trên đó gần mép mức sàn. Cùng đường cong này được xem lại ở Bài 35 từ chiều ngược lại: ở đó, năng lực được giữ ở mức cao rộng rãi và *dữ liệu* thì khan hiếm, biến "đủ năng lực để biểu diễn câu trả lời đúng" thành "cũng đủ năng lực để ghi nhớ câu trả lời sai".

Độ rộng không phải là núm điều chỉnh duy nhất. **Độ sâu (depth)** — xếp chồng thêm lớp thay vì làm một lớp rộng hơn — là trục còn lại, và nó không chỉ là "độ rộng đội lốt": một mạng sâu, hẹp có thể biểu diễn một số hàm gọn hơn nhiều so với một mạng nông, rộng cần (ít đơn vị tổng cộng hơn cho cùng khả năng biểu diễn), nhưng thay vào đó nó lại đưa vào kiểu thất bại riêng của nó. Bài 37 tiếp nối trực tiếp điều này: xếp chồng đủ nhiều lớp có thể khiến gradient biến mất trước khi chúng kịp đến được các lớp sớm nhất, một vấn đề mà mạng 2 lớp nông của bài này không gặp phải.

### Bài tập

1. Phép quét năng lực ở trên dùng ReLU. Hãy chạy lại với `tanh` thay thế (đổi hàm kích hoạt và đạo hàm của nó trong `forward`/`backward` xuyên suốt `train_and_eval`, tương tự Bài tập 2 bên dưới). Mức trần $H=1$ và bước chuyển $H=3$-sang-$H=4$ có xuất hiện như cũ không, hay một hàm kích hoạt trơn làm thay đổi vị trí của mức sàn?
2. Thay `relu`/`relu_deriv` bằng `tanh`/đạo hàm của nó ($1 - \tanh^2$) xuyên suốt, rồi huấn luyện lại. Nó có còn giải được bài toán không? So sánh hình dạng đường cong hàm mất mát thu được với phiên bản ReLU.
3. Loại bỏ hoàn toàn hàm phi tuyến (thay `relu(z1)` bằng chỉ `z1` trong `forward`, và `relu_deriv(z1)` bằng một mảng toàn số 1 trong `backward`). Xác nhận rằng mạng không còn vượt qua được mức trần ~50% của Bài 31 &mdash; nhất quán với lập luận sụp-đổ-thành-một-phép-chiếu ở trên, giờ được minh họa trên động lực huấn luyện thực tế thay vì chỉ trên đại số tĩnh.